In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import pandas as pd
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



In [ ]:
values = np.linspace(-4, 3, num=20)
C_values = 10**values

In [ ]:
# 1. Lasso (L1 penalty)
lasso_lr = LogisticRegression(penalty='l1', solver='liblinear', max_iter=10000)
lasso_grid = GridSearchCV(lasso_lr, param_grid={'C': C_values}, cv=10,n_jobs=-1)
lasso_grid.fit(X_train, y_train)


In [ ]:
# 2. Ridge (L2 penalty)
ridge_lr = LogisticRegression(penalty='l2', max_iter=10000)
ridge_grid = GridSearchCV(ridge_lr, param_grid={'C': C_values}, cv=10,n_jobs=-1)
ridge_grid.fit(X_train, y_train)


In [ ]:
# 3. Elastic Net
# Requires penalty='elasticnet' and solver='saga'
en_lr = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=10000)
en_param_grid = {
    'C': np.logspace(-4, 3, 6),
    'l1_ratio': np.linspace(0.1, 0.9, 5) # strictly between 0 and 1
}
en_grid = GridSearchCV(en_lr, param_grid=en_param_grid, cv=10,n_jobs=-1)
en_grid.fit(X_train, y_train)

In [ ]:
print(f"Lasso Best C: {lasso_grid.best_params_['C']}")
print(f"Ridge Best C: {ridge_grid.best_params_['C']}")
print(f"Elastic Net Best Params: {en_grid.best_params_}")

In [ ]:
results = [
    {
        "Model": "Lasso",
        "Best CV Score": lasso_grid.best_score_,
        "Score Std": lasso_grid.cv_results_['std_test_score'][lasso_grid.best_index_],
        "Best Params": lasso_grid.best_params_
    },
    {
        "Model": "Ridge",
        "Best CV Score": ridge_grid.best_score_,
        "Score Std": ridge_grid.cv_results_['std_test_score'][ridge_grid.best_index_],
        "Best Params": ridge_grid.best_params_
    },
    {
        "Model": "Elastic Net",
        "Best CV Score": en_grid.best_score_,
        "Score Std": en_grid.cv_results_['std_test_score'][en_grid.best_index_],
        "Best Params": en_grid.best_params_
    }
]

# Convert to DataFrame and sort by best score (descending)
cv_comparison = pd.DataFrame(results).sort_values(by="Best CV Score", ascending=False).reset_index(drop=True)

# Display the comparison table
print(cv_comparison)